In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Orquestador Maestro — AlertaFuego Cloud
# MAGIC
# MAGIC Pipeline diario automático. Ejecutar a las 06:00 UTC.
# MAGIC
# MAGIC **Flujo:**
# MAGIC 1. Extracción Open-Meteo (4 días forecast)
# MAGIC 2. Actualización ventana deslizante del seed (INSERT nuevos + DELETE viejos >35 días)
# MAGIC 3. Forecast Transform → forecast_gold_temp
# MAGIC 4. Inferencia XGBoost V2 → predictions_ui.json
# MAGIC 5. Verificación de salida
# MAGIC
# MAGIC **One-time setup** (solo la primera vez o para resetear):
# MAGIC Correr manualmente `01_landing/extract_openmeteo_seed` antes de este pipeline.

# COMMAND ----------

from datetime import datetime, timedelta
import pandas as pd

# CONFIGURACIÓN
WORKSPACE_BASE     = "/Users/jmendelewicz02@gmail.com/fire_prediction_model"
NOTEBOOK_EXTRACT   = f"{WORKSPACE_BASE}/01_landing/extract_open_meteo_forecast"
NOTEBOOK_TRANSFORM = f"{WORKSPACE_BASE}/model_v2/forecast_transform"
NOTEBOOK_INFER     = f"{WORKSPACE_BASE}/model_v2/cloud_inference_engine"

CATALOG      = "fire_risk_project"
TABLE_SEED   = f"{CATALOG}.00_landing.forecast_seed"
PATH_FORECAST = f"/Volumes/{CATALOG}/00_landing/open_meteo_forecast"
OUTPUT_JSON  = f"/Volumes/{CATALOG}/03_gold/outputs/predictions_ui.json"
SEED_WINDOW  = 35  # días a mantener en la tabla seed

# COMMAND ----------

# MAGIC %md ## Paso 1 · Extracción Open-Meteo (4 días)

# COMMAND ----------

import subprocess, sys

packages = [
    "openmeteo-requests",
    "requests-cache", 
    "retry-requests"
]

for pkg in packages:
    print(f"Instalando {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])

print("Dependencias listas.")


print("=" * 60)
print("PASO 1: EXTRACCIÓN OPEN-METEO")
print("=" * 60)
result_extract = dbutils.notebook.run(NOTEBOOK_EXTRACT, timeout_seconds=1800)
print(f"Resultado: {result_extract}")

# COMMAND ----------

# MAGIC %md ## Paso 2 · Actualizar ventana deslizante del seed

# COMMAND ----------

print("=" * 60)
print("PASO 2: ACTUALIZACIÓN SEED (ventana deslizante)")
print("=" * 60)

# Cargar el CSV recién extraído
files = dbutils.fs.ls(PATH_FORECAST)
latest = sorted([f.path for f in files if "forecast_" in f.name])[-1]
df_new = pd.read_csv(latest.replace("dbfs:/Volumes", "/Volumes").replace("dbfs:", "/dbfs"))
df_new["date"] = pd.to_datetime(df_new["date"])

# Solo nos quedamos con las fechas que ya pasaron (datos reales, no forecast)
# El forecast del día anterior ya es dato real hoy
fecha_hoy = pd.Timestamp.now().normalize()
df_insertar = df_new[df_new["date"] < fecha_hoy].copy()

if len(df_insertar) > 0:
    # INSERT nuevas filas al seed via Spark
    sdf_new = spark.createDataFrame(df_insertar)
    sdf_new.createOrReplaceTempView("nuevos_datos")

    spark.sql(f"""
        MERGE INTO {TABLE_SEED} AS target
        USING nuevos_datos AS source
        ON target.cell_id = source.cell_id AND target.date = source.date
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"  Insertadas: {len(df_insertar):,} filas ({df_insertar['date'].nunique()} días nuevos)")
else:
    print("  No hay fechas pasadas en el CSV para insertar.")

# DELETE filas con más de SEED_WINDOW días de antigüedad
fecha_corte = (fecha_hoy - timedelta(days=SEED_WINDOW)).strftime("%Y-%m-%d")
resultado_delete = spark.sql(f"""
    DELETE FROM {TABLE_SEED}
    WHERE date < '{fecha_corte}'
""")
print(f"  Eliminadas filas anteriores a: {fecha_corte}")

# Verificar estado actual del seed
seed_stats = spark.sql(f"""
    SELECT COUNT(*) as filas, 
           COUNT(DISTINCT cell_id) as nodos,
           COUNT(DISTINCT date) as dias,
           MIN(date) as desde, 
           MAX(date) as hasta
    FROM {TABLE_SEED}
""").collect()[0]
print(f"  Seed actual: {seed_stats['filas']:,} filas | {seed_stats['nodos']} nodos | {seed_stats['dias']} días ({seed_stats['desde']} → {seed_stats['hasta']})")

# COMMAND ----------

# MAGIC %md ## Paso 3 · Forecast Transform

# COMMAND ----------

print("=" * 60)
print("PASO 3: FORECAST TRANSFORM (FWI + FEATURES)")
print("=" * 60)
result_transform = dbutils.notebook.run(NOTEBOOK_TRANSFORM, timeout_seconds=1800)
print(f"Resultado: {result_transform}")

# COMMAND ----------

# MAGIC %md ## Paso 4 · Inferencia + Export JSON + Cleanup

# COMMAND ----------

print("=" * 60)
print("PASO 4: INFERENCIA Y EXPORTACIÓN")
print("=" * 60)
result_infer = dbutils.notebook.run(NOTEBOOK_INFER, timeout_seconds=900)
print(f"Resultado: {result_infer}")

# COMMAND ----------

# MAGIC %md ## Paso 5 · Verificación

# COMMAND ----------

try:
    info    = dbutils.fs.ls(OUTPUT_JSON)
    size_kb = info[0].size / 1024
    print("=" * 60)
    print("PIPELINE COMPLETADO")
    print("=" * 60)
    print(f"  Archivo  : {OUTPUT_JSON}")
    print(f"  Tamaño   : {size_kb:.1f} KB")
    print(f"  Extract  : {result_extract}")
    print(f"  Transform: {result_transform}")
    print(f"  Infer    : {result_infer}")
    print(f"  Timestamp: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M UTC')}")
except Exception as e:
    print("ERROR: No se encontró el archivo de salida.")
    raise e